# Fine-tune ResNet18 on MNIST and save weights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shuhrat-1/MLP_assignments/blob/main/assignment_4/train_mnist.ipynb)

Replace `USERNAME/REPO` in the badge URL above with your GitHub path.

**Pipeline** (same as class notebook T11):
1. Create model, train it, save weights with `torch.save(model.state_dict(), "model.pth")`.
2. Re-create the model, load the weights, predict (done on the HF Space).

The only file needed for deployment is `model.pth`.

> **Enable GPU:** Runtime -> Change runtime type -> T4 GPU. Training ResNet18 on 256x256 images on CPU is very slow.

## Part 1: Create, train, and save the model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

MNIST is grayscale (1 channel). ResNet expects 3-channel input, so we convert grayscale to 3 channels and use the standard ImageNet normalization (same transform we will reuse in the deployment app, minus augmentation).

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize(size=(256, 256)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_set = datasets.MNIST(root="./data", train=True, download=True,
                           transform=data_transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
print("Training samples:", len(train_set))

Re-create ResNet18 with pre-trained weights and replace the final fully-connected layer to output 10 classes (digits 0-9).

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(in_features=512, out_features=10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print("Model ready.")

In [ ]:
num_epochs = 2
print(f"Starting training for {num_epochs} epochs...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg:.4f}")

print("Training complete.")

Save **only the weights** (`state_dict`) - the recommended PyTorch approach. The architecture lives in code; the `.pth` file holds numbers.

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Model weights saved to model.pth")

## Part 2: Re-create the model, load weights, predict

Simulates a fresh environment (e.g. the Hugging Face Space). We only have `model.pth` and knowledge of the architecture.

In [ ]:
# Re-create the EXACT same architecture
loaded_model = resnet18(weights=None)
loaded_model.fc = nn.Linear(in_features=512, out_features=10)

# Load the saved weights
loaded_model.load_state_dict(
    torch.load("model.pth", map_location=torch.device("cpu"))
)
loaded_model.eval()  # important for BatchNorm/Dropout layers
print("Weights loaded. Model in evaluation mode.")

In [ ]:
import torch.nn.functional as F

# Grab one test image to verify the loaded model predicts correctly
test_set = datasets.MNIST(root="./data", train=False, download=True,
                          transform=data_transform)
img, true_label = test_set[0]

with torch.no_grad():
    logits = loaded_model(img.unsqueeze(0)).flatten()
    probs = F.softmax(logits, dim=0)

print("True label:", true_label)
print("Predicted :", int(probs.argmax()))
for d in range(10):
    print(f"  digit {d}: {probs[d]:.4f}")

## Part 3: Download the weights for deployment

Upload the downloaded `model.pth` to your Hugging Face Space alongside `app.py` and `requirements.txt`.

In [ ]:
from google.colab import files
files.download("model.pth")